# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、严谨的解释
- **额外要求**：用**流式（streaming）**一边生成一边显示，而不是等整段答完才一次性打印

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| 流式输出 `stream=True` | 逐 token / 逐块 `yield`，边收边 `print` |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），用 HTTP 调本地接口 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_KEY`；若要用 Llama，还需 `OLLAMA_URL`（例如 `http://localhost:11434`），并确保 Ollama 已拉取 `llama3.2`
3. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Llama 两格，对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key、Ollama 地址
import os
# 导入标准库 requests：用 HTTP 请求调用本地 Ollama 的聊天接口
import requests
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 typing 导入类型标注工具：Optional（可空）、Literal（只允许若干固定字符串）
from typing import Optional, Literal
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import display, Markdown


In [32]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'


In [33]:
# ========== 环境 + Model 类：一个类里封装「问 GPT」和「问 Llama」两条路径 ==========

# 加载 .env：把 OPENAI_KEY、OLLAMA_URL 等读入进程环境（不写进笔记本正文）
load_dotenv()

class Model:
    """封装两个后端：OpenAI（SDK）与 Ollama（HTTP），统一用 prompt() 流式提问。"""

    def __init__(self):
        # 创建 OpenAI 客户端；密钥从环境变量 OPENAI_KEY 读取（注意：本练习用的名字是 OPENAI_KEY，不是常见的 OPENAI_API_KEY）
        self.client_oai = OpenAI(api_key=os.getenv("OPENAI_KEY"))
        # 拼出 Ollama 原生聊天 API 地址：{OLLAMA_URL}/api/chat（流式 NDJSON，不是 OpenAI 兼容的 /v1）
        self.ollama_base_url = f"{os.getenv('OLLAMA_URL')}/api/chat"

    def _prompt_llama(self, text: str):
        """私有方法：向本地 Ollama 发流式聊天请求，并逐行 yield 原始响应文本。"""
        # POST 到 Ollama /api/chat；json 里 stream=True 表示服务端持续推送；requests 侧 stream=True 表示边下边读
        response = requests.post(
            self.ollama_base_url,
            json={
                "model": MODEL_LLAMA,
                "messages": [{"role": "user", "content": text}],
                "stream": True
            },
            stream=True
        )
        # iter_lines：按行迭代响应体（Ollama 流式通常一行一个 JSON）
        for line in response.iter_lines():
            # 空行跳过
            if not line:
                continue

            # 字节 → 字符串
            data = line.decode("utf-8")
            if data.strip() == "":
                continue

            # 兼容可能出现的 SSE 前缀 "data:"（有的代理/网关会加）
            if data.startswith("data:"):
                data = data[5:].strip()

            # yield：把这一行交给外层；本函数是生成器（generator），调用方用 for 慢慢取
            yield data

    def _prompt_oai(self, question: str):
        """私有方法：用 OpenAI SDK 发起流式 Chat Completions，返回可迭代的 stream 对象。"""
        stream = self.client_oai.chat.completions.create(
            model=MODEL_GPT,
            messages=[
                {
                    "role": "system",
                    # system prompt 保留英文：这是发给模型的指令，改译会改变回答风格/行为
                    "content": (
                        "You are an advanced reasoning and explanation engine. "
                        "You write with precision, clarity, and conciseness. "
                        "You can explain Python, algorithms, code design, and systems-level behavior "
                        "with technical rigor, while being straight to the point."
                    ),
                },
                # user：真正的用户问题
                {"role": "user", "content": question},
            ],
            # stream=True：不要等整段生成完，而是持续返回增量 delta
            stream=True,
        )
        return stream

    def prompt(self, question: str, model: Optional[Literal[MODEL_GPT, MODEL_LLAMA]] = MODEL_GPT):
        """统一入口：按 model 名字分流到 GPT 或 Llama，流式 yield 文本块，最后再 display 一次完整 Markdown。"""
        # 简单分流：名字里含 "gpt" 走 OpenAI，否则走 Ollama
        if "gpt" in model:
            stream = self._prompt_oai(question)
            # buffer：把每一块拼起来，最后用于整段 Markdown 展示
            buffer = []
            for event in stream:
                # 流式事件里，增量文本在 choices[0].delta.content
                if event.choices and event.choices[0].delta.content:
                    chunk = event.choices[0].delta.content
                    buffer.append(chunk)
                    # 立刻 yield 给调用方，便于边生成边 print
                    yield chunk

            output = "".join(buffer)

        else:
            stream = self._prompt_llama(question)
            buffer = []
            for chunk in stream:
                try:
                    # 每行是 JSON 字符串；解析后从 message.content 取出增量文本
                    import json
                    data = json.loads(chunk)
                    content = data.get("message", {}).get("content", "")

                    if content:
                        buffer.append(content)
                        yield content

                except Exception as e:
                    # 某行解析失败就打印错误并继续，避免整次流式中断
                    print("An error occured", e)
                    continue

            output = "".join(buffer)

        # 全部收齐后，用 Markdown 在笔记本里再漂亮显示一遍完整回答
        display(Markdown(output))


In [25]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 把技术问题写在三引号字符串里；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码，再分别跑下面 GPT / Llama 两格做对比
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [27]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# 实例化封装好的 Model（内部会读环境变量、创建 OpenAI 客户端）
model = Model()

# model.prompt 是生成器：每来一块 token/文本就进入循环
# end="" 表示不换行拼接；flush=True 表示立刻刷到屏幕，实现「打字机」效果
for token in model.prompt(question, model=MODEL_GPT):
    print(token, end="", flush=True)


The provided code snippet is a Python expression that uses `yield from` with a set comprehension. Here’s a breakdown of what it does:

1. **Set Comprehension**: The expression `{book.get("author") for book in books if book.get("author")}` creates a set of unique authors extracted from a list (or any iterable) called `books`. 
    - `book.get("author")` retrieves the value associated with the key `"author"` from each `book` dictionary.
    - The `if book.get("author")` condition filters out any books that do not have an author (i.e., where the author value is `None` or an empty string). 

2. **Yield from**: The `yield from` statement is used in a generator function to yield all values from the iterable that follows it. In this case, it yields the unique authors produced by the set comprehension.

### Why Use This Code?

- **Uniqueness**: Using a set comprehension ensures that only unique authors are collected, automatically eliminating duplicates.
- **Efficiency**: The code succinctly p

The provided code snippet is a Python expression that uses `yield from` with a set comprehension. Here’s a breakdown of what it does:

1. **Set Comprehension**: The expression `{book.get("author") for book in books if book.get("author")}` creates a set of unique authors extracted from a list (or any iterable) called `books`. 
    - `book.get("author")` retrieves the value associated with the key `"author"` from each `book` dictionary.
    - The `if book.get("author")` condition filters out any books that do not have an author (i.e., where the author value is `None` or an empty string). 

2. **Yield from**: The `yield from` statement is used in a generator function to yield all values from the iterable that follows it. In this case, it yields the unique authors produced by the set comprehension.

### Why Use This Code?

- **Uniqueness**: Using a set comprehension ensures that only unique authors are collected, automatically eliminating duplicates.
- **Efficiency**: The code succinctly processes the list of books and yields only the relevant data (authors) directly from the generator, making it memory-efficient and lazy.
- **Readability**: The use of `yield from` keeps the code clean and avoids the need to create an intermediate list before yielding authors.

### Example:

Given a list of books represented as dictionaries:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},
    {"title": "Book 4", "author": None},
]
```

The code would yield `Author A` and `Author B`, iterating over each author only once.

### Conclusion:

The code succinctly generates a lazily yielded sequence of unique authors from a collection of book dictionaries, efficiently handling potential duplicates and missing values.

In [34]:
# ========== 路径 B：用本地 Llama 3.2（Ollama）流式回答 ==========

# 复用同一个 Model 实例；只把 model= 换成 MODEL_LLAMA
# 理念：同一问题、两个后端 —— 对比云端 API 与本地开源模型的速度、风格、是否需要密钥
# 前提：Ollama 在跑，且已安装 llama3.2；.env 里 OLLAMA_URL 正确
for token in model.prompt(question, model=MODEL_LLAMA):
    print(token, end="", flush=True)


This line of code is a part of Python's iteration protocol, specifically using the `yield from` keyword.

Here's what it does:

- It generates an iterator that yields values from another iterable.
- The expression `{book.get("author") for book in books if book.get("author")}` generates an iterator over the authors of all books where an author is available (`book.get("author") != None or book.get("author") == ""`).

The `yield from` keyword takes this generator expression and yields from it. It's essentially saying "use this generator to generate values, I'll take them in order".

Here's a step-by-step explanation:

1. `{book.get("author") for book in books if book.get("author")}` generates an iterator over the authors of all books where an author is available.

   - This works by iterating over each `book` in the collection (`books`), checking if it has a valid author, and then yielding the author's name.
   - If `book.get("author")` returns `None`, or if its value is an empty string, 

This line of code is a part of Python's iteration protocol, specifically using the `yield from` keyword.

Here's what it does:

- It generates an iterator that yields values from another iterable.
- The expression `{book.get("author") for book in books if book.get("author")}` generates an iterator over the authors of all books where an author is available (`book.get("author") != None or book.get("author") == ""`).

The `yield from` keyword takes this generator expression and yields from it. It's essentially saying "use this generator to generate values, I'll take them in order".

Here's a step-by-step explanation:

1. `{book.get("author") for book in books if book.get("author")}` generates an iterator over the authors of all books where an author is available.

   - This works by iterating over each `book` in the collection (`books`), checking if it has a valid author, and then yielding the author's name.
   - If `book.get("author")` returns `None`, or if its value is an empty string, it skips that book.

2. `yield from { ... }` takes this generator expression and yields from it.

   - It's like saying "take all values yielded by the inner generator and use them one by one". 

   This makes the code cleaner and easier to read because you don't have to manually call `next(book_generator)` for each book in the collection. You can simply iterate over this new iterator to get all authors.

However, without more context about how these books and their data are structured, it's hard to tell exactly why or when someone would use this line of code specifically. But generally, it's useful for generating values from other iterables while still following a clear sequence.

Here's an example:

```python
books = [
    {"title": "Book 1", "author": None},
    {"title": "Book 2", "author": "Author 1"},
    {"title": "Book 3", "author": ""}
]

for author in yield from {book.get("author") for book in books if book.get("author")}:
    print(author)
```

This would print:

```
Author 1
```